# JMIR evidence-selection experiment — linguistic repair, replacement, and validation

Repairs the 124 usable evaluation cases, replaces invalid C023 with one unused VQA-RAD yes/no case, and produces a 100 + 25 review-ready evaluation draft. It does not freeze or run inference.

## 0. Runtime prerequisite
Use one T4 runtime. Run the Ollama installation and server cells from Notebook 03 first; no second GPU session is needed.

In [5]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,os,random,re,subprocess,time,urllib.request
import pandas as pd
from PIL import Image
ROOT=Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK_DIR=ROOT/'casebooks'; CASEBOOK_DIR.mkdir(parents=True,exist_ok=True)
IMAGE_DIR=ROOT/'images'/'vqa_rad'; IMAGE_DIR.mkdir(parents=True,exist_ok=True)
drafts=list(ROOT.rglob('candidate_casebook_evaluation_machine_draft.csv'))
if not drafts: raise FileNotFoundError('Upload the Notebook 04 output under the project directory.')
SOURCE=max(drafts,key=lambda p:p.stat().st_mtime)
df=pd.read_csv(SOURCE,dtype=str,keep_default_na=False)
assert len(df)==135 and len(df[df.phase.eq('evaluation')])==125
print('Source:',SOURCE)
print('Statuses:',df.review_status.value_counts().to_dict())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Source: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook_evaluation_machine_draft.csv
Statuses: {'machine_drafted': 96, 'replacement_required': 29, 'approved': 10}


## 1. Replace C023 deterministically from unused VQA-RAD cases

In [6]:
# subprocess.run(['pip','-q','install','-U','datasets','pillow','pyarrow','pandas==2.2.3'],check=True)
from datasets import load_dataset
vqa=load_dataset('flaviagiammarino/vqa-rad')
def image_fingerprint(image,fallback):
    if isinstance(image,Image.Image):
        x=image.convert('RGB');return hashlib.sha1((str(x.size)+'|RGB|').encode()+x.tobytes()).hexdigest()[:16]
    if isinstance(image,dict) and image.get('bytes'):return hashlib.sha1(image['bytes']).hexdigest()[:16]
    return hashlib.sha1(str(fallback).encode()).hexdigest()[:16]
rows=[]
for split,part in vqa.items():
  for i,x in enumerate(part):
    q=str(x.get('question') or '').strip();a=str(x.get('answer') or '').strip().lower().rstrip('.')
    if a not in {'yes','no'} or re.match(r'^(what|which|who|where|when|why|how)\b',q,re.I):continue
    image=x.get('image');name=str(x.get('image_name') or x.get('image_id') or f'{split}_{i}')
    rows.append({'split':split,'source_index':i,'image_key':image_fingerprint(image,name),'image_name':name,'question_en':q,'reference_answer':a,'answer_type_raw':str(x.get('answer_type') or ''),'image_object':image})
pool=pd.DataFrame(rows)
used=set(df.loc[df.case_id.ne('C023'),'image_key'])
pool=pool[~pool.image_key.isin(used)].drop_duplicates('image_key')
# Preserve the original 50/50 balance: C023 carried a Yes reference.
pool=pool[pool.reference_answer.eq('yes')].sample(frac=1,random_state=20260921+99).reset_index(drop=True)
replacement=pool.iloc[0]
idx=df.index[df.case_id.eq('C023')][0]
for c in ['split','source_index','image_key','image_name','question_en','reference_answer','answer_type_raw']:df.at[idx,c]=str(replacement[c])
df.at[idx,'dataset']='vqa_rad';df.at[idx,'phase']='evaluation';df.at[idx,'smoke_test']='False'
path=IMAGE_DIR/'C023.jpg';replacement.image_object.convert('RGB').save(path,quality=95)
df.at[idx,'image_path']='images/vqa_rad/C023.jpg';df.at[idx,'image_saved']='True'
for c in ['question_id','question_ms','proposition_en','proposition_id','proposition_ms']:df.at[idx,c]=''
df.at[idx,'clinical_validity_notes']='Replacement for original open-ended C023; unused binary VQA-RAD case selected deterministically.'
df.at[idx,'review_status']='replacement_drafted'
print('Replacement question:',df.at[idx,'question_en'],'Answer:',df.at[idx,'reference_answer'],'Image:',path)

Replacement question: is this film properly exposed? Answer: yes Image: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/images/vqa_rad/C023.jpg


In [8]:
import subprocess
import time
import urllib.request

# Locate Ollama
result = subprocess.run(
    ["bash", "-lc", "command -v ollama"],
    capture_output=True,
    text=True
)

OLLAMA = result.stdout.strip()
print("Ollama binary:", OLLAMA or "NOT FOUND")

if not OLLAMA:
    raise RuntimeError("Ollama installation did not succeed.")

# Start server if necessary
def ollama_alive():
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )
        return True
    except Exception:
        return False

if not ollama_alive():
    ollama_server = subprocess.Popen(
        [OLLAMA, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for _ in range(60):
        if ollama_alive():
            break
        time.sleep(1)

if not ollama_alive():
    raise RuntimeError("Ollama was installed but its server did not start.")

print("Ollama is installed and running.")
subprocess.run([OLLAMA, "list"], check=True)

Ollama binary: NOT FOUND


RuntimeError: Ollama installation did not succeed.

In [9]:
# Install Ollama correctly on a Google Colab x86_64 runtime

import json
import os
import platform
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

if platform.machine() not in ("x86_64", "amd64"):
    raise RuntimeError(
        f"Unexpected architecture: {platform.machine()}. "
        "This cell expects a standard Colab x86_64 runtime."
    )

# Obtain metadata for the latest official GitHub release
api_url = "https://api.github.com/repos/ollama/ollama/releases/latest"

request = urllib.request.Request(
    api_url,
    headers={"User-Agent": "Google-Colab"}
)

with urllib.request.urlopen(request, timeout=60) as response:
    release = json.load(response)

print("Release:", release["tag_name"])

assets = release.get("assets", [])

# Exact preferred asset names, in order
preferred_names = [
    "ollama-linux-amd64.tar.zst",
    "ollama-linux-amd64.tgz",
]

selected_asset = None

for preferred_name in preferred_names:
    selected_asset = next(
        (
            asset for asset in assets
            if asset["name"].lower() == preferred_name
        ),
        None
    )
    if selected_asset:
        break

if selected_asset is None:
    available_linux_assets = [
        asset["name"]
        for asset in assets
        if "linux" in asset["name"].lower()
    ]
    raise RuntimeError(
        "The standard Linux AMD64 Ollama package was not found.\n"
        f"Available Linux assets: {available_linux_assets}"
    )

asset_name = selected_asset["name"]
download_url = selected_asset["browser_download_url"]
archive_path = Path("/content") / asset_name

print("Selected asset:", asset_name)
print("Downloading Ollama...")

subprocess.run(
    [
        "curl", "-fL",
        "--retry", "5",
        "--retry-delay", "5",
        "--connect-timeout", "30",
        "-o", str(archive_path),
        download_url
    ],
    check=True
)

print(
    f"Downloaded: {archive_path} "
    f"({archive_path.stat().st_size / 1024**2:.1f} MB)"
)

# Install extraction dependency when necessary
if asset_name.endswith(".tar.zst"):
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "zstd"],
        check=True
    )

# Extract into /usr so that bin/ollama becomes /usr/bin/ollama
print("Extracting Ollama...")

if asset_name.endswith(".tar.zst"):
    subprocess.run(
        [
            "tar",
            "--use-compress-program=unzstd",
            "-xf", str(archive_path),
            "-C", "/usr"
        ],
        check=True
    )
elif asset_name.endswith(".tgz"):
    subprocess.run(
        [
            "tar",
            "-xzf", str(archive_path),
            "-C", "/usr"
        ],
        check=True
    )
else:
    raise RuntimeError(f"Unsupported archive format: {asset_name}")

# Search expected and fallback locations
candidate_paths = [
    Path("/usr/bin/ollama"),
    Path("/usr/local/bin/ollama"),
    Path("/usr/lib/ollama/ollama"),
]

ollama_path = next(
    (path for path in candidate_paths if path.is_file()),
    None
)

if ollama_path is None:
    search_result = subprocess.run(
        ["find", "/usr", "-type", "f", "-name", "ollama"],
        capture_output=True,
        text=True,
        check=False
    )

    found_paths = [
        Path(line.strip())
        for line in search_result.stdout.splitlines()
        if line.strip()
    ]

    ollama_path = next(
        (path for path in found_paths if path.is_file()),
        None
    )

if ollama_path is None:
    raise RuntimeError(
        "The correct archive was downloaded and extracted, "
        "but the Ollama executable could not be located."
    )

ollama_path.chmod(0o755)

# Make it available at a conventional location
if ollama_path != Path("/usr/local/bin/ollama"):
    target = Path("/usr/local/bin/ollama")

    if target.exists() or target.is_symlink():
        target.unlink()

    target.symlink_to(ollama_path)

OLLAMA = "/usr/local/bin/ollama"

print("Ollama installed at:", OLLAMA)
subprocess.run([OLLAMA, "--version"], check=True)

Release: v0.34.2
Selected asset: ollama-linux-amd64.tar.zst
Downloaded: /content/ollama-linux-amd64.tar.zst (1361.4 MB)
Extracting Ollama...
Ollama installed at: /usr/local/bin/ollama


CompletedProcess(args=['/usr/local/bin/ollama', '--version'], returncode=0)

## 2. Connect to Ollama and load the two repair models
Qwen3 drafts neutral English propositions. TranslateGemma is translation-specific and translates both questions and propositions into Indonesian and Malay. Neither evaluated VLM is used.

In [10]:
import json
import os
import subprocess
import time
import urllib.request
from pathlib import Path

# --------------------------------------------------
# Locate the installed Ollama executable
# --------------------------------------------------

possible_binaries = [
    Path("/usr/bin/ollama"),
    Path("/usr/local/bin/ollama"),
]

ollama_binary = next(
    (
        str(path)
        for path in possible_binaries
        if path.exists() and os.access(path, os.X_OK)
    ),
    None,
)

if ollama_binary is None:
    raise RuntimeError(
        "Ollama is no longer installed in this runtime. "
        "Rerun the Ollama installation cells from Notebook 03, "
        "then rerun this cell."
    )

print("Ollama binary:", ollama_binary)

# --------------------------------------------------
# API helpers
# --------------------------------------------------

def ollama_alive():
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=2,
        )
        return True
    except Exception:
        return False


def api_json(path, payload=None, timeout=900):
    data = (
        None
        if payload is None
        else json.dumps(payload).encode("utf-8")
    )

    request = urllib.request.Request(
        "http://127.0.0.1:11434" + path,
        data=data,
        headers={"Content-Type": "application/json"},
    )

    with urllib.request.urlopen(
        request,
        timeout=timeout,
    ) as response:
        return json.loads(response.read())


# --------------------------------------------------
# Start Ollama if the restart stopped it
# --------------------------------------------------

if not ollama_alive():
    log_path = Path("/content/ollama_gpu_server.log")
    log_handle = open(log_path, "a")

    server_environment = os.environ.copy()
    server_environment["OLLAMA_HOST"] = "127.0.0.1:11434"

    ollama_process = subprocess.Popen(
        [ollama_binary, "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=server_environment,
    )

    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)

if not ollama_alive():
    log_path = Path("/content/ollama_gpu_server.log")

    if log_path.exists():
        print("Ollama server log:")
        print(log_path.read_text(errors="replace"))

    raise RuntimeError(
        "Ollama is installed but the server could not start."
    )

print("Ollama server ready.")

# --------------------------------------------------
# Confirm GPU visibility
# --------------------------------------------------

gpu_check = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

if gpu_check.returncode == 0:
    print("GPU:", gpu_check.stdout.strip())
else:
    print(
        "Warning: NVIDIA GPU was not detected. "
        "Check that the Colab runtime is still set to T4 GPU."
    )

# --------------------------------------------------
# Pull the proposition model
# --------------------------------------------------

OLLAMA = ollama_binary
PROP_MODEL = "qwen3:4b"
TRANS_MODEL = "translategemma:4b"

subprocess.run(
    [OLLAMA, "pull", PROP_MODEL],
    check=True,
)

print("Proposition model ready:", PROP_MODEL)

Ollama binary: /usr/bin/ollama
Ollama server ready.
GPU: Tesla T4, 15360 MiB
Proposition model ready: qwen3:4b


## 3. Redraft all English propositions with polarity safeguards

In [13]:
# Section 3 — Redraft English propositions with thinking disabled

PROP_CP = CASEBOOK_DIR / "evaluation_repair_propositions.jsonl"

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

def load_cp(path):
    out = {}

    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                x = json.loads(line)
                out[x["case_id"]] = x
            except Exception:
                pass

    return out

def parse_obj(text):
    text = str(text or "").strip()

    if not text:
        raise ValueError("Model returned empty content")

    # First try parsing the complete response
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback for responses containing surrounding text
    match = re.search(r"\{.*\}", text, re.S)

    if not match:
        raise ValueError(
            f"No JSON object found in response: {text[:300]!r}"
        )

    return json.loads(match.group())

SYSTEM = """
Create a neutral English noun phrase for a binary medical-image
question.

The phrase must represent the proposition that is TRUE when the
answer is Yes and FALSE when the answer is No.

It must fit after both:

"Evidence of [phrase] is present"

and

"Evidence of [phrase] is absent."

Preserve anatomy, laterality, modality, visibility, normality, and
negation. Never replace a general finding with a specific diagnosis.

Examples:

"Is the image normal?"
-> "normal image appearance"

"Are the lungs free of consolidation?"
-> "absence of pulmonary consolidation"

"Is the appendix non-enlarged?"
-> "a non-enlarged appendix"

"Is there a clavicle fracture?"
-> "a clavicle fracture"

Return JSON only using this structure:

{
  "proposition": "noun phrase",
  "polarity_check": "brief explanation"
}

/no_think
""".strip()

props = load_cp(PROP_CP)
evaluation = df[df["phase"].eq("evaluation")].copy()

print("Evaluation cases:", len(evaluation))
print("Existing checkpoint records:", len(props))

for n, (_, r) in enumerate(evaluation.iterrows(), 1):

    # Skip only genuinely successful records.
    # Earlier error records will automatically be retried.
    existing = props.get(r.case_id, {})

    if existing.get("status") == "ok":
        print(
            f"[{n}/{len(evaluation)}] "
            f"{r.case_id} skip: "
            f"{existing['proposition_en']}"
        )
        continue

    payload = {
        "model": PROP_MODEL,
        "stream": False,

        # Disable Qwen 3's extended thinking output
        "think": False,

        "format": {
            "type": "object",
            "properties": {
                "proposition": {
                    "type": "string"
                },
                "polarity_check": {
                    "type": "string"
                }
            },
            "required": [
                "proposition",
                "polarity_check"
            ]
        },

        "messages": [
            {
                "role": "system",
                "content": SYSTEM
            },
            {
                "role": "user",
                "content": (
                    f"Question: {str(r.question_en).strip()}\n"
                    "Return only the requested JSON object.\n"
                    "/no_think"
                )
            }
        ],

        "options": {
            "temperature": 0,
            "seed": 20260921,
            "num_predict": 300
        },

        "keep_alive": "30m"
    }

    try:
        response = api_json(
            "/api/chat",
            payload,
            timeout=900
        )

        message = response.get("message", {})
        raw_content = message.get("content", "")

        x = parse_obj(raw_content)

        proposition = str(
            x.get("proposition", "")
        ).strip().rstrip(".")

        polarity_check = str(
            x.get("polarity_check", "")
        ).strip()

        if not proposition:
            raise ValueError("Empty proposition in returned JSON")

        if re.search(r"\b(yes|no)\b", proposition, re.I):
            raise ValueError(
                f"Invalid proposition containing yes/no: "
                f"{proposition!r}"
            )

        rec = {
            "case_id": r.case_id,
            "status": "ok",
            "proposition_en": proposition,
            "polarity_check": polarity_check,
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

    except Exception as e:
        rec = {
            "case_id": r.case_id,
            "status": "error",
            "error": repr(e),
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        # Store limited diagnostic information
        if "response" in locals():
            rec["response_diagnostic"] = {
                "done_reason": response.get("done_reason"),
                "message_keys": list(
                    response.get("message", {}).keys()
                ),
                "content_preview": str(
                    response.get("message", {}).get(
                        "content", ""
                    )
                )[:500],
                "thinking_preview": str(
                    response.get("message", {}).get(
                        "thinking", ""
                    )
                )[:500]
            }

    append_jsonl(PROP_CP, rec)
    props[r.case_id] = rec

    print(
        f"[{n}/{len(evaluation)}] "
        f"{r.case_id} "
        f"{rec.get('proposition_en', rec.get('error'))}"
    )

successful_ids = {
    case_id
    for case_id, record in props.items()
    if record.get("status") == "ok"
}

evaluation_ids = set(evaluation["case_id"])

successful_evaluation_ids = successful_ids & evaluation_ids
failed_ids = sorted(evaluation_ids - successful_evaluation_ids)

print()
print(
    "Successful propositions:",
    len(successful_evaluation_ids),
    "/",
    len(evaluation_ids)
)

if failed_ids:
    print("Failed case IDs:", failed_ids)

    first_failure = props.get(failed_ids[0], {})
    print(
        "First failure diagnostic:",
        json.dumps(
            first_failure,
            ensure_ascii=False,
            indent=2
        )
    )

    raise RuntimeError(
        f"{len(failed_ids)} proposition cases remain unsuccessful. "
        "Inspect the diagnostic printed above."
    )

# Unload the proposition model before loading TranslateGemma
api_json(
    "/api/generate",
    {
        "model": PROP_MODEL,
        "keep_alive": 0
    }
)

print("All propositions successfully checkpointed.")

Evaluation cases: 125
Existing checkpoint records: 125
[1/125] C011 enhancing lesion
[2/125] C012 a pneumothorax
[3/125] C013 calcifications of the right lung
[4/125] C014 enlarged ventricles
[5/125] C015 normal x-ray appearance
[6/125] C016 bilateral involvement
[7/125] C017 a wedge-shaped lesion
[8/125] C018 a pleural effusion
[9/125] C019 abnormal image appearance
[10/125] C020 diffuse cerebral edema
[11/125] C021 a fracture on the left 5th rib
[12/125] C022 intravenous contrast use
[13/125] C023 proper film exposure
[14/125] C024 multiple septations within the cyst
[15/125] C025 abnormal hypodense liver lesions
[16/125] C026 a well circumscribed mass
[17/125] C027 liver visibility
[18/125] C028 ectatic or aneurysmal enlargement of the abdominal aorta
[19/125] C029 fluid in the anterior abdominal wall
[20/125] C030 lateral ventricles
[21/125] C031 abnormal findings in the lower lung fields
[22/125] C032 symmetry of the patient's ribs
[23/125] C033 visible sulci
[24/125] C034 a skull

## 4. Translate with TranslateGemma and checkpoint every case

In [14]:
subprocess.run([OLLAMA,'pull',TRANS_MODEL],check=True)
TRANS_CP=CASEBOOK_DIR/'evaluation_repair_translations.jsonl';translations=load_cp(TRANS_CP)
def translate_pair(question,proposition,target_name,target_code):
    prompt=f'''You are a professional English (en) to {target_name} ({target_code}) medical translator. Translate the two fields faithfully. Preserve medical terminology, laterality, negation, modality names, and the difference between heart and liver. “Appendix” means the anatomical appendix, not a document attachment. Return strict JSON only with keys question and proposition.\nQUESTION: {question}\nPROPOSITION: {proposition}'''
    payload={'model':TRANS_MODEL,'stream':False,'format':'json','messages':[{'role':'user','content':prompt}],'options':{'temperature':0,'seed':20260921,'num_predict':180}}
    x=parse_obj(api_json('/api/chat',payload)['message']['content']);return str(x['question']).strip(),str(x['proposition']).strip().rstrip('.')
for n,(_,r) in enumerate(evaluation.iterrows(),1):
    if r.case_id in translations and translations[r.case_id].get('status')=='ok':print(f'[{n}/125] skip',r.case_id);continue
    p=props[r.case_id]['proposition_en']
    try:
        qid,pid=translate_pair(r.question_en,p,'Indonesian','id');qms,pms=translate_pair(r.question_en,p,'Malay','ms')
        rec={'case_id':r.case_id,'status':'ok','question_id':qid,'proposition_id':pid,'question_ms':qms,'proposition_ms':pms,'timestamp_utc':datetime.now(timezone.utc).isoformat()}
    except Exception as e:rec={'case_id':r.case_id,'status':'error','error':repr(e)}
    append_jsonl(TRANS_CP,rec);translations[r.case_id]=rec;print(f'[{n}/125]',r.case_id,rec['status'])
assert sum(x.get('status')=='ok' for x in translations.values())>=125,'Rerun to retry errors'
print('All translations checkpointed.')

[1/125] C011 ok
[2/125] C012 ok
[3/125] C013 ok
[4/125] C014 ok
[5/125] C015 ok
[6/125] C016 ok
[7/125] C017 ok
[8/125] C018 ok
[9/125] C019 ok
[10/125] C020 ok
[11/125] C021 ok
[12/125] C022 ok
[13/125] C023 ok
[14/125] C024 ok
[15/125] C025 ok
[16/125] C026 ok
[17/125] C027 ok
[18/125] C028 ok
[19/125] C029 ok
[20/125] C030 ok
[21/125] C031 ok
[22/125] C032 ok
[23/125] C033 ok
[24/125] C034 ok
[25/125] C035 ok
[26/125] C036 ok
[27/125] C037 ok
[28/125] C038 ok
[29/125] C039 ok
[30/125] C040 ok
[31/125] C041 ok
[32/125] C042 ok
[33/125] C043 ok
[34/125] C044 ok
[35/125] C045 ok
[36/125] C046 ok
[37/125] C047 ok
[38/125] C048 ok
[39/125] C049 ok
[40/125] C050 ok
[41/125] C051 ok
[42/125] C052 ok
[43/125] C053 ok
[44/125] C054 ok
[45/125] C055 ok
[46/125] C056 ok
[47/125] C057 ok
[48/125] C058 ok
[49/125] C059 ok
[50/125] C060 ok
[51/125] C061 ok
[52/125] C062 ok
[53/125] C063 ok
[54/125] C064 ok
[55/125] C065 ok
[56/125] C066 ok
[57/125] C067 ok
[58/125] C068 ok
[59/125] C069 ok
[60/12

## 5. Automated terminology, completeness, and polarity audit

In [15]:
out=df.copy();flags=[]
for idx,r in out[out.phase.eq('evaluation')].iterrows():
    p=props[r.case_id];t=translations[r.case_id]
    out.at[idx,'proposition_en']=p['proposition_en']
    for c in ['question_id','question_ms','proposition_id','proposition_ms']:out.at[idx,c]=t[c]
    rowflags=[];combined=' | '.join([t['question_id'],t['question_ms'],t['proposition_id'],t['proposition_ms']]).lower()
    if not all(str(out.at[idx,c]).strip() for c in ['question_en','question_id','question_ms','proposition_en','proposition_id','proposition_ms']):rowflags.append('missing_field')
    if 'heart' in r.question_en.lower() and ('hati' in t['question_id'].lower() or 'hati' in t['question_ms'].lower()):rowflags.append('heart_liver_confusion')
    if 'appendix' in r.question_en.lower() and ('lampiran' in combined):rowflags.append('anatomical_appendix_mistranslated')
    if 'clavicle' in r.question_en.lower() and ('tengkorak' in combined):rowflags.append('clavicle_skull_confusion')
    if 'pleural effusion' in r.question_en.lower() and any(x in combined for x in ['pendarahan','pembekuan']):rowflags.append('effusion_mistranslated')
    if re.search(r'\bnormal\b|free of|non-enlarged',r.question_en,re.I) and not re.search(r'normal|absence|free|non-enlarged',p['proposition_en'],re.I):rowflags.append('polarity_review')
    flags.append({'case_id':r.case_id,'flags':';'.join(rowflags)})
    out.at[idx,'include_after_review']=''
    out.at[idx,'reviewer_initials']=''
    out.at[idx,'review_status']='review_ready' if not rowflags else 'needs_manual_review'
    out.at[idx,'clinical_validity_notes']='Independent proposition drafting and TranslateGemma repair completed.'+(' Flags: '+';'.join(rowflags) if rowflags else '')
flagdf=pd.DataFrame(flags);display(flagdf[flagdf['flags'].ne('')])
print('Status counts:',out.review_status.value_counts().to_dict())

,case_id,flags


Status counts: {'review_ready': 125, 'approved': 10}


## 6. Verify allocation and save the review-ready CSV

In [18]:
# Repair missing SLAKE images from the dataset's separate imgs.zip archive

from huggingface_hub import hf_hub_download
from io import BytesIO
from pathlib import PurePosixPath
import zipfile

SLAKE_IMAGE_DIR = ROOT / "images" / "slake"
SLAKE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Identify the 25 SLAKE evaluation rows
slake_mask = (
    out["phase"].astype(str).str.strip().str.lower().eq("evaluation")
    &
    out["dataset"].astype(str).str.strip().str.lower().eq("slake")
)

slake_rows = out.loc[slake_mask].copy()

print("SLAKE evaluation rows:", len(slake_rows))

assert len(slake_rows) == 25, (
    f"Expected 25 SLAKE evaluation rows, "
    f"found {len(slake_rows)}."
)

# Download the official SLAKE image archive.
# It is cached in the Colab runtime, not copied to Google Drive.
print("Downloading/accessing official SLAKE imgs.zip...")

slake_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset"
)

print("Archive:", slake_zip_path)

saved_count = 0
failures = []

with zipfile.ZipFile(slake_zip_path, "r") as archive:

    # Retain only actual files
    archive_members = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    print("Files in archive:", len(archive_members))

    # Normalize archive member names
    normalized_members = {
        name.replace("\\", "/").lstrip("./"): name
        for name in archive_members
    }

    for idx, row in slake_rows.iterrows():

        case_id = str(row["case_id"]).strip()

        wanted_name = (
            str(row["image_name"])
            .replace("\\", "/")
            .lstrip("./")
            .strip()
        )

        # Common possible archive layouts
        direct_candidates = [
            wanted_name,
            f"imgs/{wanted_name}",
            f"SLAKE/{wanted_name}",
            f"SLAKE/imgs/{wanted_name}",
        ]

        matched_member = None

        # Try exact normalized matches first
        for candidate in direct_candidates:
            candidate = candidate.replace("\\", "/").lstrip("./")

            if candidate in normalized_members:
                matched_member = normalized_members[candidate]
                break

        # Otherwise locate a member ending in the stored relative path
        if matched_member is None:
            suffix_matches = [
                original_name
                for normalized_name, original_name
                in normalized_members.items()
                if (
                    normalized_name == wanted_name
                    or normalized_name.endswith("/" + wanted_name)
                )
            ]

            if len(suffix_matches) == 1:
                matched_member = suffix_matches[0]
            elif len(suffix_matches) > 1:
                # Prefer the shortest path if duplicate wrappers exist
                matched_member = sorted(
                    suffix_matches,
                    key=lambda x: len(
                        PurePosixPath(x).parts
                    )
                )[0]

        if matched_member is None:
            failures.append({
                "case_id": case_id,
                "image_name": wanted_name,
                "reason": "not found in imgs.zip"
            })
            print(
                f"[FAILED] {case_id}: "
                f"{wanted_name} not found"
            )
            continue

        destination = SLAKE_IMAGE_DIR / f"{case_id}.jpg"

        try:
            image_bytes = archive.read(matched_member)

            with Image.open(BytesIO(image_bytes)) as image:
                image.convert("RGB").save(
                    destination,
                    format="JPEG",
                    quality=95
                )

            # Verify that PIL can reopen the saved file
            with Image.open(destination) as verification_image:
                verification_image.verify()

            relative_path = (
                Path("images") /
                "slake" /
                f"{case_id}.jpg"
            )

            out.at[idx, "image_path"] = str(relative_path)
            out.at[idx, "image_saved"] = "true"

            # Keep df synchronized in case it is used later
            matching_df = df.index[
                df["case_id"].astype(str).eq(case_id)
            ]

            if len(matching_df) == 1:
                df_idx = matching_df[0]
                df.at[df_idx, "image_path"] = str(relative_path)
                df.at[df_idx, "image_saved"] = "true"

            saved_count += 1

            print(
                f"[{saved_count}/25] {case_id}: "
                f"{wanted_name} -> {destination.name}"
            )

        except Exception as e:
            failures.append({
                "case_id": case_id,
                "image_name": wanted_name,
                "archive_member": matched_member,
                "reason": repr(e)
            })

            print(
                f"[FAILED] {case_id}: {repr(e)}"
            )

print()
print("SLAKE images saved:", saved_count, "/ 25")

if failures:
    display(pd.DataFrame(failures))

    raise RuntimeError(
        f"{len(failures)} SLAKE images could not be restored."
    )

# Final physical-file verification
missing_files = []

for idx, row in out.loc[slake_mask].iterrows():

    image_path = ROOT / str(row["image_path"])

    if not image_path.is_file():
        missing_files.append({
            "case_id": row["case_id"],
            "expected_path": str(image_path)
        })
        continue

    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception as e:
        missing_files.append({
            "case_id": row["case_id"],
            "expected_path": str(image_path),
            "error": repr(e)
        })

if missing_files:
    display(pd.DataFrame(missing_files))
    raise RuntimeError(
        "Some SLAKE image files failed physical verification."
    )

print("All 25 SLAKE image files physically verified.")
print(
    "Updated image-saved counts:",
    out.loc[
        out["phase"].astype(str).str.lower().eq("evaluation"),
        "image_saved"
    ].astype(str).str.lower().value_counts().to_dict()
)

SLAKE evaluation rows: 25
Downloading/accessing official SLAKE imgs.zip...


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

Archive: /root/.cache/huggingface/hub/datasets--BoKelvin--SLAKE/snapshots/a9083ce6c34ac3ffb17671a605962924d8a8f9e9/imgs.zip
Files in archive: 2592
[1/25] C111: xmlab436/source.jpg -> C111.jpg
[2/25] C112: xmlab273/source.jpg -> C112.jpg
[3/25] C113: xmlab113/source.jpg -> C113.jpg
[4/25] C114: xmlab161/source.jpg -> C114.jpg
[5/25] C115: xmlab563/source.jpg -> C115.jpg
[6/25] C116: xmlab562/source.jpg -> C116.jpg
[7/25] C117: xmlab79/source.jpg -> C117.jpg
[8/25] C118: xmlab437/source.jpg -> C118.jpg
[9/25] C119: xmlab245/source.jpg -> C119.jpg
[10/25] C120: xmlab48/source.jpg -> C120.jpg
[11/25] C121: xmlab405/source.jpg -> C121.jpg
[12/25] C122: xmlab25/source.jpg -> C122.jpg
[13/25] C123: xmlab549/source.jpg -> C123.jpg
[14/25] C124: xmlab473/source.jpg -> C124.jpg
[15/25] C125: xmlab360/source.jpg -> C125.jpg
[16/25] C126: xmlab182/source.jpg -> C126.jpg
[17/25] C127: xmlab551/source.jpg -> C127.jpg
[18/25] C128: xmlab591/source.jpg -> C128.jpg
[19/25] C129: xmlab116/source.jpg -> 

In [19]:
# Section 6 — Verify allocation and save review-ready CSV

# Normalize fields before validation
out["phase"] = (
    out["phase"]
    .astype(str)
    .str.strip()
    .str.lower()
)

out["dataset"] = (
    out["dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
)

out["reference_answer"] = (
    out["reference_answer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.rstrip(".")
)

out["image_saved"] = (
    out["image_saved"]
    .astype(str)
    .str.strip()
    .str.lower()
)

ev = out[out["phase"].eq("evaluation")].copy()

dataset_counts = ev["dataset"].value_counts().to_dict()
answer_counts = ev["reference_answer"].value_counts().to_dict()
image_counts = ev["image_saved"].value_counts().to_dict()

print("Evaluation rows:", len(ev))
print("Dataset counts:", dataset_counts)
print("Answer counts:", answer_counts)
print("Image-saved values:", image_counts)

# 1. Total allocation
assert len(ev) == 125, (
    f"Expected 125 evaluation cases, found {len(ev)}."
)

# 2. Dataset allocation
assert dataset_counts == {
    "vqa_rad": 100,
    "slake": 25
}, (
    "Unexpected dataset allocation: "
    f"{dataset_counts}"
)

# 3. Ensure only valid binary references are present
unexpected_answers = set(answer_counts) - {"yes", "no"}

assert not unexpected_answers, (
    "Unexpected reference-answer values: "
    f"{sorted(unexpected_answers)}"
)

assert answer_counts.get("yes", 0) + answer_counts.get("no", 0) == 125, (
    "Some evaluation cases do not have a valid yes/no reference. "
    f"Counts: {answer_counts}"
)

# 4. Accept either orientation of a balanced odd-number allocation:
#    63 yes / 62 no OR 62 yes / 63 no
assert abs(
    answer_counts.get("yes", 0) -
    answer_counts.get("no", 0)
) <= 1, (
    "The evaluation set is not balanced between yes and no. "
    f"Counts: {answer_counts}"
)

# 5. Confirm every image is saved
assert ev["image_saved"].eq("true").all(), (
    "Some evaluation images are not marked as saved. "
    f"Counts: {image_counts}"
)

# 6. Confirm all generated language fields are populated
required_text_fields = [
    "question_en",
    "question_id",
    "question_ms",
    "proposition_en",
    "proposition_id",
    "proposition_ms"
]

missing_summary = {}

for column in required_text_fields:
    missing_mask = (
        ev[column].isna() |
        ev[column].astype(str).str.strip().eq("")
    )

    missing_count = int(missing_mask.sum())

    if missing_count:
        missing_summary[column] = missing_count

assert not missing_summary, (
    "Some required language fields remain empty: "
    f"{missing_summary}"
)

# 7. Confirm that case IDs are unique
duplicate_case_ids = (
    ev.loc[
        ev["case_id"].duplicated(keep=False),
        "case_id"
    ]
    .astype(str)
    .tolist()
)

assert not duplicate_case_ids, (
    "Duplicate evaluation case IDs found: "
    f"{duplicate_case_ids}"
)

# Save durable review-ready output
OUTPUT = (
    CASEBOOK_DIR /
    "candidate_casebook_evaluation_review_ready.csv"
)

out.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8"
)

sha = hashlib.sha256(
    OUTPUT.read_bytes()
).hexdigest()

checksum_path = OUTPUT.with_suffix(".csv.sha256")

checksum_path.write_text(
    f"{sha}  {OUTPUT.name}\n",
    encoding="utf-8"
)

print()
print("All allocation and completeness checks passed.")
print("Saved:", OUTPUT)
print("SHA256:", sha)
print(
    "Evaluation allocation:",
    ev.groupby(
        ["dataset", "reference_answer"]
    ).size().to_dict()
)

display(
    out.loc[
        out["phase"].eq("evaluation"),
        [
            "case_id",
            "dataset",
            "question_en",
            "question_id",
            "question_ms",
            "proposition_en",
            "proposition_id",
            "proposition_ms",
            "review_status",
            "clinical_validity_notes"
        ]
    ].head(12)
)

Evaluation rows: 125
Dataset counts: {'vqa_rad': 100, 'slake': 25}
Answer counts: {'no': 63, 'yes': 62}
Image-saved values: {'true': 125}

All allocation and completeness checks passed.
Saved: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook_evaluation_review_ready.csv
SHA256: 032c8f4cc70b22e95fc87c11c983f7c6d9af3723ba42261e31dc2c02a15d113c
Evaluation allocation: {('slake', 'no'): 13, ('slake', 'yes'): 12, ('vqa_rad', 'no'): 50, ('vqa_rad', 'yes'): 50}


,case_id,dataset,question_en,question_id,question_ms,proposition_en,proposition_id,proposition_ms,review_status,clinical_validity_notes
10,C011,vqa_rad,is the lesion enhancing?,apakah lesi tersebut mengalami peningkatan?,apakah lesi tersebut menunjukkan peningkatan?,enhancing lesion,lesi yang meningkat,lesi yang menunjukkan peningkatan,review_ready,Independent proposition drafting and Translate...
11,C012,vqa_rad,is a pneuomothorax present?,¿Apakah terdapat pneumotoraks?,apakah terdapat pneumothorax?,a pneumothorax,pneumotoraks,pneumothorax,review_ready,Independent proposition drafting and Translate...
12,C013,vqa_rad,are these small opacities in the right lung ca...,¿Apakah opasitas kecil ini di paru kanan adala...,apakah kawasan bayangan kecil ini di paru-paru...,calcifications of the right lung,kalsifikasi pada paru kanan,calcifications di paru-paru kanan,review_ready,Independent proposition drafting and Translate...
13,C014,vqa_rad,are the ventricles enlarged?,¿Apakah ventrikel membesar?,Adakah bilik jantung membesar?,enlarged ventricles,Ventrikel membesar,Bilik jantung yang membesar,review_ready,Independent proposition drafting and Translate...
14,C015,vqa_rad,is this a normal x ray?,¿Apakah ini radiografi normal?,apakah ini adalah sinar-x yang normal?,normal x-ray appearance,tampak radiografi normal,penampilan sinar-x yang normal,review_ready,Independent proposition drafting and Translate...
15,C016,vqa_rad,were both sides affected?,apakah kedua sisi terpengaruh?,apakah kedua belah sisi terbabit?,bilateral involvement,terlibatnya kedua sisi,keterlibatan kedua belah,review_ready,Independent proposition drafting and Translate...
16,C017,vqa_rad,is the lesion wedge-shaped?,¿Apakah lesi berbentuk seperti segitiga?,apakah lesi berbentuk seperti segitiga?,a wedge-shaped lesion,lesi berbentuk seperti segitiga,lesi berbentuk seperti segitiga,review_ready,Independent proposition drafting and Translate...
17,C018,vqa_rad,is a pleural effusion present?,apakah terdapat efusi pleura?,Adakah kehadiran cairan di ruang pleura?,a pleural effusion,efusi pleura,Kehadiran cairan di ruang pleura,review_ready,Independent proposition drafting and Translate...
18,C019,vqa_rad,is this picture abnormal?,Apakah gambar ini abnormal?,Adakah imej ini tidak normal?,abnormal image appearance,Tampak abnormal,penampilan imej yang tidak normal,review_ready,Independent proposition drafting and Translate...
19,C020,vqa_rad,do we see diffuse cerebral edema in this mri?,¿Apakah kita melihat edema serebral difus pada...,Adakah edema serebral yang tersebar dalam imej...,diffuse cerebral edema,edema serebral difus,Edema serebral yang tersebar,review_ready,Independent proposition drafting and Translate...


## Stop here
Upload `candidate_casebook_evaluation_review_ready.csv`. Do not change statuses to `approved` or freeze the evaluation until the final row-by-row audit is complete.